# 02 Domain Shift: Hybrid Vs Paired

This notebook shows why naive transfer was never going to be enough. The central question is not whether the model is powerful, but whether the source and target domains are even close enough for direct transfer to work.

**Questions answered here**
- How separable are hybrid and paired units?
- Does context make prediction easier while also making domains more separable?
- Which feature groups carry the strongest hybrid-to-paired shift?


In [ ]:
from pathlib import Path
import sys
import json

import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

def _find_thesis_root():
    cwd = Path.cwd().resolve()
    direct = [cwd, *cwd.parents]
    nested = [candidate / "thesis" for candidate in direct]
    for candidate in [*direct, *nested]:
        if (
            (candidate / "src" / "qc_thesis" / "__init__.py").exists()
            and (candidate / "README.md").exists()
            and (candidate / "notebooks").exists()
        ):
            return candidate
    raise FileNotFoundError("Could not find thesis root from notebook session")

ROOT = _find_thesis_root()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from qc_thesis import *

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)
apply_thesis_style()


def show_saved_figure(path, caption=None):
    figure_path = Path(path)
    if not figure_path.is_absolute():
        figure_path = (ROOT / figure_path).resolve()
    if not figure_path.exists():
        display(Markdown(f"_Missing figure: `{figure_path}`_"))
        return
    if caption:
        display(Markdown(caption))
    display(Image(filename=str(figure_path)))

fig_dir, table_dir = notebook_output_dirs("02_domain_shift_hybrid_vs_paired")
live_bundle = get_domain_shift_bundle()
frozen_domain_summary = pd.read_csv(ROOT / "data/frozen_inputs/domain_shift/domain_classifier_summary.csv")
frozen_feature_group = pd.read_csv(ROOT / "data/frozen_inputs/domain_shift/feature_group_summary.csv")


## 1. Domain separability

The benchmark result that matters here is simple: if a lightweight classifier can tell hybrid from paired almost perfectly, then naive transfer is structurally unsafe. Context usually helps the prediction task, but it also increases domain identifiability.


In [ ]:
display(frozen_domain_summary)
save_table(frozen_domain_summary, table_dir, "domain_classifier_summary")
fig, _ = plot_domain_shift_separability(live_bundle.domain_summary)
save_figure(fig, fig_dir, "domain_separability")
fig


## 2. Which feature groups shift the most?

We want to know whether the shift lives only in broad recording context, or whether unit-level morphology is also shifted. The answer is: both matter, but context blocks and waveform-related features are especially strong.


In [ ]:
display(frozen_feature_group.head(12))
display(live_bundle.feature_shift.head(20))
save_table(frozen_feature_group, table_dir, "feature_group_shift_summary")
save_table(live_bundle.feature_shift.head(50), table_dir, "feature_shift_summary_top50")
fig, _ = plot_top_feature_shift(live_bundle.feature_shift, top_n=20)
save_figure(fig, fig_dir, "top_feature_shift")
fig


## 3. PCA structure

PCA does not prove transfer failure by itself, but it makes the geometry visible. Hybrid and paired units occupy different regions, and the paired families are not collapsed onto one compact manifold.


In [ ]:
fig, _ = plot_pca_projection(live_bundle.pca_2d, color_col="dataset_type")
save_figure(fig, fig_dir, "pca_2d_by_domain")
fig


In [ ]:
paired_only = live_bundle.pca_2d[live_bundle.pca_2d["dataset_type"] == "paired"].copy()
centroids = (
    paired_only.groupby("study_set", as_index=False)
    .agg(pc1_mean=("pc1", "mean"), pc2_mean=("pc2", "mean"), n_rows=("recording_key", "size"), n_recordings=("recording_key", "nunique"))
    .sort_values("study_set")
    .reset_index(drop=True)
)
display(centroids)
save_table(centroids, table_dir, "paired_family_pca_centroids")
fig, _ = plot_pca_projection(paired_only, color_col="study_set")
save_figure(fig, fig_dir, "pca_2d_paired_by_family")
fig


In [ ]:
fig, _ = plot_pca_projection_3d(live_bundle.pca_3d, color_col="dataset_type")
save_figure(fig, fig_dir, "pca_3d_by_domain")
fig


In [ ]:
unit_auc = float(frozen_domain_summary.query("feature_set == 'unit'").iloc[0]["cv_auc_mean"])
ctx_auc = float(frozen_domain_summary.query("feature_set == 'full_context'").iloc[0]["cv_auc_mean"])
top_group = frozen_feature_group.sort_values("mean_abs_smd", ascending=False).iloc[0]["feature_group"]
display(Markdown(
    f"""
## Key takeaways

- Hybrid vs paired separation is already very high on unit features (**AUC {unit_auc:.3f}**).
- With full context, the domain classifier becomes essentially perfect (**AUC {ctx_auc:.3f}**).
- The strongest shifted feature block in the frozen audit is **{top_group}**, which helps explain why context is both useful and potentially shortcut-prone.
- This notebook motivates the rest of the thesis: the task is not “train a stronger regressor”, it is “transfer under severe dataset shift”.
"""
))


---
### Thesis highlight — Domain shift story

_Provenance._ This figure is a thesis synthesis artifact regenerated by `scripts/make_thesis_highlight_figures.py` from frozen summary tables. It deliberately combines three views that would otherwise live in separate exploratory figures:

- domain-classifier AUC on unit vs full-context feature sets
- full-context feature-group shift magnitudes
- paired-family PCA centroids

The point is not that one feature group alone “explains” the transfer gap. The narrower thesis-safe claim is that **the hybrid→paired gap is severe and the paired target domain is internally structured rather than one compact regime**.


In [ ]:
show_saved_figure(
    ROOT / "figures" / "09_thesis_highlight_figures" / "domain_shift_story.png",
    "This synthesis figure is regenerated from frozen summary tables before the notebooks are rebuilt, so the notebook does not depend on a hand-edited PNG.",
)
